# MITSUI Commodity Prediction — Colab training and submission

End-to-end training, validation and sequential inference for all 424 targets.
The notebook uses the uploaded competition notebook as its blueprint: target
pair features, LightGBM, Random Forest, XGBoost, stacking and the Kaggle
inference server. The implementation fixes label alignment, future leakage and
in-sample stacking while retaining that model architecture.

## Blueprint mapping

| Uploaded notebook section | Repository implementation |
|---|---|
| `get_data_for_day` and generated targets | official labels aligned by `date_id` |
| `prepare_features_for_col/df` | `src/mitsui/features.py` |
| LGBM + RF + XGB base learners | `src/mitsui/ensemble.py` |
| XGB meta-model trained in-sample | time-series OOF Ridge meta-model |
| `predict_on_test` | `src/mitsui/inference.py` |
| `MitsuiInferenceServer` | `scripts/run_local_gateway.py` and final cells |

The original negative shifts and backward fill are not retained because they
read future rows. All positive lag, rolling, difference/spread concepts remain
in causal form.

## 1. Clone and install

In [5]:
!git clone https://github.com/mingzhuoFUN/mitsui-commodity-prediction.git
%cd mitsui-commodity-prediction
!pip -q install -r requirements.txt
%env PYTHONPATH=src

Cloning into 'mitsui-commodity-prediction'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 55 (delta 13), reused 46 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (55/55), 30.16 KiB | 5.03 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/mitsui-commodity-prediction/mitsui-commodity-prediction/mitsui-commodity-prediction
env: PYTHONPATH=src


## 2. Competition data

Recommended authentication: add the current `KGAT_...` token to Colab Secrets
with the name `KAGGLE_API_TOKEN` and enable notebook access. A legacy
`kaggle.json` upload remains available as a fallback. Never place either
credential in GitHub or a notebook cell.

In [6]:
from pathlib import Path
import os
DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not (DATA_DIR / "train.csv").exists():
    try:
        from google.colab import userdata
        api_token = userdata.get("KAGGLE_API_TOKEN")
    except Exception:
        api_token = None

    if api_token:
        os.environ["KAGGLE_API_TOKEN"] = api_token
    else:
        from google.colab import files
        print("No KAGGLE_API_TOKEN secret found; upload legacy kaggle.json.")
        uploaded = files.upload()
        token_file = next(iter(uploaded))
        kaggle_dir = Path.home() / ".kaggle"
        kaggle_dir.mkdir(exist_ok=True)
        (kaggle_dir / "kaggle.json").write_bytes(uploaded[token_file])
        os.chmod(kaggle_dir / "kaggle.json", 0o600)
    !kaggle competitions download -c mitsui-commodity-prediction-challenge -p data/raw
    !unzip -q -o data/raw/mitsui-commodity-prediction-challenge.zip -d data/raw

!python scripts/inspect_data.py --data-dir data/raw

100% 10.4M/10.4M [00:00<00:00, 56.6MB/s]

Data inventory:
                                                          path  size_mb suffix
                                 kaggle_evaluation/__init__.py    0.001    .py
                            kaggle_evaluation/core/__init__.py    0.000    .py
                        kaggle_evaluation/core/base_gateway.py    0.019    .py
                  kaggle_evaluation/core/generated/__init__.py    0.000    .py
     kaggle_evaluation/core/generated/kaggle_evaluation_pb2.py    0.004    .py
kaggle_evaluation/core/generated/kaggle_evaluation_pb2_grpc.py    0.003    .py
                kaggle_evaluation/core/kaggle_evaluation.proto    0.002 .proto
                               kaggle_evaluation/core/relay.py    0.017    .py
                           kaggle_evaluation/core/templates.py    0.005    .py
                           kaggle_evaluation/mitsui_gateway.py    0.003    .py
                  kaggle_evaluation/mitsui_inference_server.py    0.000  

## 3. Verify official X/Y alignment and target horizons

In [7]:
import pandas as pd
train = pd.read_csv(DATA_DIR / "train.csv")
labels = pd.read_csv(DATA_DIR / "train_labels.csv")
pairs = pd.read_csv(DATA_DIR / "target_pairs.csv")
assert train["date_id"].equals(labels["date_id"])
display(pairs.groupby("lag").size().rename("targets"))
print("market", train.shape, "labels", labels.shape)

,targets
lag,
1,106
2,106
3,106
4,106


market (1961, 558) labels (1961, 425)


## 4. Leakage checks

Tests include a future-mutation test: changing future market rows must not
change features already computed for past rows.

In [8]:
!pytest -q

.....                                                                    [100%]
5 passed in 2.55s


In [13]:
!pip -q install -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for mitsui-reproduction (pyproject.toml) ... done


In [16]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))

## 5. Inspect the actual model code

The notebook calls repository modules so the same tested implementation is
used in Colab, local validation and Kaggle inference.

In [18]:
from mitsui.features import make_target_features
from mitsui.ensemble import fit_stacked_models, predict_stacked_models
from mitsui.inference import SequentialPredictor

print("Base models: LightGBM, Random Forest, XGBoost")
print("Meta model: Ridge trained from time-series OOF predictions")

Base models: LightGBM, Random Forest, XGBoost
Meta model: Ridge trained from time-series OOF predictions


## 6. Eight-target smoke run

In [19]:
!python scripts/train_ensemble.py \
  --data-dir data/raw \
  --output-dir outputs/ensemble_smoke \
  --valid-size 128 \
  --max-targets 8

{
  "n_models": 8,
  "fit_full": false,
  "ic_sharpe": 0.12973141603315005,
  "mean_daily_ic": 0.06577622919086333,
  "std_daily_ic": 0.507018509487753
}


## 7. Full chronological validation

Each target uses LightGBM, Random Forest and XGBoost. Their time-series OOF
predictions train a Ridge meta-model. The final 252 rows are untouched until
validation.

In [20]:
!python scripts/train_ensemble.py \
  --data-dir data/raw \
  --output-dir outputs/ensemble_full \
  --valid-size 252

{
  "n_models": 424,
  "fit_full": false,
  "ic_sharpe": 0.20124308818116868,
  "mean_daily_ic": 0.04370396587139754,
  "std_daily_ic": 0.21717002191922805
}


In [21]:
import json
json.loads(Path("outputs/ensemble_full/metrics.json").read_text())

{'n_models': 424,
 'fit_full': False,
 'ic_sharpe': 0.20124308818116868,
 'mean_daily_ic': 0.04370396587139754,
 'std_daily_ic': 0.21717002191922805}

## 8. Fit submission models on all official training rows

In [22]:
!python scripts/train_ensemble.py \
  --data-dir data/raw \
  --output-dir outputs/ensemble_submit \
  --fit-full

{
  "n_models": 424,
  "fit_full": true
}


## 9. Run the complete local Kaggle gateway

In [23]:
!python scripts/run_local_gateway.py \
  --data-dir data/raw \
  --model-path outputs/ensemble_submit/stacked_models.pkl

## 10. Competition rerun entrypoint

For a Kaggle submission, keep the trained model artifact in a Kaggle Dataset
attached to the notebook, initialize `SequentialPredictor`, then serve it:

In [24]:
import os, sys
import pandas as pd
from mitsui.ensemble import load_stacked_models
from mitsui.inference import SequentialPredictor

sys.path.append(str(DATA_DIR.resolve()))
from kaggle_evaluation.mitsui_inference_server import MitsuiInferenceServer

predictor = SequentialPredictor(
    load_stacked_models("outputs/ensemble_submit/stacked_models.pkl"),
    pd.read_csv(DATA_DIR / "train.csv"),
    pd.read_csv(DATA_DIR / "train_labels.csv"),
)

def predict(test, label_lags_1, label_lags_2, label_lags_3, label_lags_4):
    return predictor.predict(
        test, label_lags_1, label_lags_2, label_lags_3, label_lags_4
    )

inference_server = MitsuiInferenceServer(predict)
if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    inference_server.serve()
else:
    print("Use the previous cell for local gateway validation.")

Use the previous cell for local gateway validation.


In [25]:
import json
from pathlib import Path

metrics = json.loads(
    Path("outputs/ensemble_full/metrics.json").read_text()
)

metrics

{'n_models': 424,
 'fit_full': False,
 'ic_sharpe': 0.20124308818116868,
 'mean_daily_ic': 0.04370396587139754,
 'std_daily_ic': 0.21717002191922805}

In [26]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [27]:
!mkdir -p "/content/drive/MyDrive/mitsui-model"

In [28]:
!cp "outputs/ensemble_submit/stacked_models.pkl" \
    "/content/drive/MyDrive/mitsui-model/stacked_models.pkl"

In [29]:
!cp "outputs/ensemble_full/metrics.json" \
    "/content/drive/MyDrive/mitsui-model/metrics.json"

!cp "outputs/ensemble_full/validation_predictions.csv" \
    "/content/drive/MyDrive/mitsui-model/validation_predictions.csv"

In [30]:
from pathlib import Path

save_dir = Path("/content/drive/MyDrive/mitsui-model")

for file in save_dir.iterdir():
    print(file.name, round(file.stat().st_size / 1024**2, 2), "MB")

stacked_models.pkl 132.43 MB
metrics.json 0.0 MB
validation_predictions.csv 2.3 MB


## Artifacts

`outputs/ensemble_submit/stacked_models.pkl` contains all 424 target bundles.
Store it in Google Drive or a private Kaggle Dataset; it is intentionally not
committed to GitHub.